# Scenario 03: MCP Tool Integration

**QE Perspective:** OGX can invoke tools exposed via the Model Context Protocol (MCP). This notebook validates:

- Starting in-process MCP servers with a simple `add` tool over **both** transports (Streamable HTTP and SSE).
- Calling the OGX Responses API with `tools=[{"type": "mcp", ...}]` so the model discovers and invokes the tool.
- The agent calls the tool and synthesizes the result into a response.

Configuration: `BASE_URL` / `INFERENCE_MODEL`. Server assumed at `http://localhost:8321`.

Optional overrides: `MCP_HOST` (default `127.0.0.1`), `MCP_PORT` (default 8322, SSE uses MCP_PORT+1), `MCP_SERVER_URL` (default `http://localhost:{MCP_PORT}` — override for podman/container setups where OGX can't reach host localhost).

## Setup

Load base URL and model from environment; create the OGX client.

In [ ]:
import os
import scripts.helpers  # noqa: F401 — loads .env if present

base_url = os.environ.get("OGX_BASE_URL") or os.environ.get(
    "BASE_URL", "http://localhost:8321"
)
base_url = base_url.rstrip("/")
if base_url.endswith("/v1"):
    base_url = base_url[:-3].rstrip("/")
model = os.environ.get("INFERENCE_MODEL")

MCP_HOST = os.environ.get("MCP_HOST", "0.0.0.0")
MCP_PORT = int(os.environ.get("MCP_PORT", "8322"))
mcp_server_url = os.environ.get("MCP_SERVER_URL", f"http://localhost:{MCP_PORT}")

assert base_url, "OGX_BASE_URL or BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

from ogx_client import OgxClient

client = OgxClient(base_url=base_url)
print(f"OGX: {base_url}, Model: {model}, MCP server: {mcp_server_url}")

## Define and start MCP servers

Define a minimal MCP tool (`add`) and start two servers — one using **Streamable HTTP** (the MCP spec standard) and one using **SSE** (legacy transport still used by upstream tests). This validates that OGX can consume MCP tools over both transports.

In [ ]:
import hashlib
import threading
import time
import urllib.error
import urllib.request

from mcp.server.fastmcp import FastMCP

MCP_SSE_PORT = MCP_PORT + 1


def _make_mcp_app(name, host, port):
    app = FastMCP(name, host=host, port=port)

    @app.tool()
    def get_project_info(field: str) -> str:
        """Look up internal project information by field name."""
        seed = f"ogx-mcp-{field}"
        return hashlib.sha256(seed.encode()).hexdigest()[:8]

    return app


def _wait_for_server(port, path, label):
    for _ in range(20):
        try:
            urllib.request.urlopen(f"http://localhost:{port}{path}", timeout=1)
            return True
        except urllib.error.HTTPError:
            return True
        except (urllib.error.URLError, OSError):
            time.sleep(0.5)
    raise AssertionError(f"MCP {label} server failed to start on port {port} after 10s")


# Streamable HTTP server on MCP_PORT (/mcp endpoint)
_app_http = _make_mcp_app("mcp-http", MCP_HOST, MCP_PORT)
threading.Thread(
    target=_app_http.run, kwargs={"transport": "streamable-http"}, daemon=True
).start()
_wait_for_server(MCP_PORT, "/mcp", "streamable-http")
print(f"Streamable HTTP server running on {MCP_HOST}:{MCP_PORT}")

# SSE server on MCP_SSE_PORT (/sse endpoint)
_app_sse = _make_mcp_app("mcp-sse", MCP_HOST, MCP_SSE_PORT)
threading.Thread(target=_app_sse.run, kwargs={"transport": "sse"}, daemon=True).start()
_wait_for_server(MCP_SSE_PORT, "/sse", "sse")
print(f"SSE server running on {MCP_HOST}:{MCP_SSE_PORT}")

# Pre-compute expected answer so we can assert it
EXPECTED_ANSWER = hashlib.sha256(b"ogx-mcp-codename").hexdigest()[:8]
print(f"Expected tool result for 'codename': {EXPECTED_ANSWER}")

## Request with MCP tool (both transports)

Call the Responses API with `tools=[{"type": "mcp", ...}]` pointing at each MCP server. OGX connects, discovers the `add` tool, and the model invokes it to compute 2 + 3. We test both Streamable HTTP and SSE transports.

In [ ]:
mcp_endpoints = {
    "streamable-http": f"{mcp_server_url}/mcp",
    "sse": f"{mcp_server_url.replace(str(MCP_PORT), str(MCP_SSE_PORT))}/sse",
}

for transport_name, endpoint in mcp_endpoints.items():
    print(f"\n{'=' * 40}")
    print(f"Testing MCP over {transport_name}: {endpoint}")
    print(f"{'=' * 40}")

    response = client.responses.create(
        model=model,
        input="What is the project codename? Use the get_project_info tool with field 'codename' and tell me the exact result.",
        tools=[
            {
                "type": "mcp",
                "server_label": f"mcp-{transport_name.replace('streamable-', '')}",
                "server_url": endpoint,
                "require_approval": "never",
                "allowed_tools": ["get_project_info"],
            }
        ],
        stream=False,
    )

    assert response.status == "completed", (
        f"[{transport_name}] Expected status completed, got {response.status}"
    )
    assert response.output is not None, (
        f"[{transport_name}] Expected output to be present"
    )

    has_tool_usage = False
    full_text = ""
    for item in response.output:
        item_type = getattr(item, "type", None)
        if item_type == "message" and getattr(item, "content", None):
            for c in item.content:
                if getattr(c, "text", None):
                    full_text += c.text
        if item_type in ("function_call", "mcp_call") or getattr(
            item, "tool_calls", None
        ):
            has_tool_usage = True

    if getattr(response, "output_text", None):
        full_text = full_text or response.output_text

    assert EXPECTED_ANSWER in full_text, (
        f"[{transport_name}] Expected hash '{EXPECTED_ANSWER}' in response "
        f"(proves tool was actually called), got: {full_text[:200]}"
    )
    print(f"  Tool usage: {has_tool_usage}")
    print(f"  Response contains '{EXPECTED_ANSWER}': YES")
    print(f"  Response: {full_text[:200]}")
    print("  PASSED")

print("\nAll MCP transports verified.")

## QE Assertions summary

- Two MCP servers start successfully: Streamable HTTP on `MCP_PORT`, SSE on `MCP_PORT+1`.
- OGX connects to each server and discovers the `get_project_info` tool.
- For each transport:
  - Response status is `completed`.
  - Output is present.
  - The response contains the exact hash value that only the tool can produce — proving the model actually called the tool and didn't just guess the answer.